# Análise exploratória da base de churn

**Disciplina:** Sistemas Inteligentes — CIn/UFPE  
**Etapa atual:** Partes 1 e 2 — contextualização e compreensão inicial.

Este notebook registra a investigação da base disponibilizada no miniprojeto.
As etapas seguintes estão planejadas, mas aguardam revisão antes da implementação.


## 1. Introdução e contextualização

### 1.1. Objetivo da análise

A Análise Exploratória de Dados (EDA) busca compreender o conteúdo, a estrutura e
as características da base antes de definir tratamentos ou utilizar algoritmos.
Nesta primeira entrega, o objetivo é carregar os dados corretamente, reconhecer
os atributos e apresentar um resumo descritivo inicial.

### 1.2. Base de dados e problema de churn

*Churn* corresponde ao cancelamento ou encerramento do relacionamento de um
cliente com um serviço. Estudar esse evento pode ajudar a compreender quais
características estão associadas à saída de clientes.

O arquivo disponibilizado é
`base_churn_5000_VF2_aluno.xlsx - Dados_Sinteticos_VF2.csv`.
Embora o nome contenha `.xlsx`, seu formato efetivo é **CSV**, com cabeçalho,
campos separados por vírgulas, codificação UTF-8 e vírgula decimal na mensalidade.
Campos que contêm vírgulas internas aparecem entre aspas.

Os registros contêm identificação do cliente, aquisição do serviço, mensalidade,
chamados, atrasos, contrato, pagamento e data de ocorrência de churn.
As descrições dos atributos neste notebook são preliminares, inferidas de seus
nomes e dos valores observados; não foi fornecido um dicionário de dados local.
O número de linhas será obtido pela leitura, sem assumir o valor sugerido pelo nome do arquivo.

### 1.3. Relação com as próximas etapas do projeto

A EDA fornecerá evidências para decidir o pré-processamento posteriormente.
Nos projetos supervisionados, será necessário definir e validar o alvo de churn.
Nos não supervisionados, a compreensão dos atributos ajudará a escolher quais
informações podem descrever perfis de clientes.

Nesta etapa, não serão aplicados tratamentos, criadas variáveis preditoras ou
treinados modelos. O arquivo original será preservado e a análise será feita em memória.


### 1.4. Roteiro completo proposto para a EDA

| Parte | Conteúdo e justificativa | Situação |
|---|---|---|
| 1. Introdução e contextualização | Apresentar o problema, os objetivos e os limites da análise. | Implementada |
| 2. Carregamento e compreensão inicial | Ler a base, observar dimensões, exemplos, tipos e estatísticas iniciais. | Implementada |
| 3. Qualidade e consistência dos dados | Investigar ausências, repetições, categorias, domínios e coerência das datas; distinguir ausência de informação de ausência de evento, sem corrigir registros. | Planejada |
| 4. Análise univariada | Examinar distribuições numéricas e frequências categóricas; investigar valores extremos com gráficos e resumos justificados, sem removê-los. | Planejada |
| 5. Compreensão da variável-alvo | Confirmar o significado de churn, a data de referência e a janela de observação antes de examinar as classes e seu equilíbrio. | Planejada |
| 6. Análise bivariada | Comparar atributos numéricos e categóricos com o churn, considerando o tamanho dos grupos e sem interpretar associação como causa. | Planejada |
| 7. Correlações entre atributos | Investigar associações numéricas e possível redundância, respeitando o significado de indicadores e contagens. | Planejada |
| 8. Investigações complementares | Responder apenas a perguntas motivadas pelas evidências anteriores; considerar relacionamento e chamados se houver definição temporal suficiente. | Planejada |
| 9. Síntese e encaminhamentos | Consolidar achados, limitações e recomendações para o pré-processamento futuro. | Planejada |

A ordem parte do entendimento geral para análises mais específicas, como na
referência Lígia. A auditoria de consistência temporal também é pertinente à
referência de churn. A separação entre dados originais, análise e tratamento é
adotada conceitualmente, mantendo o trabalho atual neste notebook.

**Referências consultadas:**

- [Análise de churn — branch feature/churn-eda-preprocessing](https://github.com/llucasEmanuel/CIN0024-churn-analysis-system/tree/feature/churn-eda-preprocessing), especialmente `notebooks/EDA.ipynb`.
- [Projeto Lígia — Grupo 10](https://github.com/victoriapessoabm/PS_Ligia_Grupo_10), especialmente `Notebooks/EDA.ipynb` e a organização em `Data`, `Notebooks` e `src`.

As referências orientam a organização. Seus resultados, regras de domínio e
janelas temporais não são assumidos como válidos para esta base.


## 2. Carregamento e compreensão inicial

### 2.1. Bibliotecas e configuração

Utilizamos Pandas para leitura e tabelas, NumPy para apoio numérico, e Matplotlib
e Seaborn para as visualizações das próximas partes. Aqui, tabelas são suficientes
para a inspeção inicial. As configurações abaixo alteram apenas a apresentação.

Execute as células na ordem apresentada, com o interpretador `.venv/bin/python`
e o diretório de trabalho na pasta deste notebook.


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams.update({"figure.figsize": (9, 5), "axes.titlesize": 13})

print("Notebook funcionando!")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")


Notebook funcionando!
Pandas: 2.3.3 | NumPy: 2.2.6


### 2.2. Leitura do arquivo original

`read_csv` respeita as aspas dos campos com vírgulas internas e utiliza
`decimal=","` para interpretar corretamente os valores monetários. Isso é uma
configuração de leitura, sem alterar ou regravar o arquivo.

As datas permanecem em sua representação textual. Em particular, `data_chamado`
pode reunir várias datas em uma única célula, portanto não deve ser interpretada
como uma data única. Nenhuma conversão temporal ou limpeza será realizada agora.
Campos vazios são representados pelo Pandas como `NaN`; seu significado será
investigado na etapa de qualidade.


In [2]:
caminho_base = Path("base_churn_5000_VF2_aluno.xlsx - Dados_Sinteticos_VF2.csv")

if not caminho_base.is_file():
    raise FileNotFoundError(
        "Arquivo não encontrado. Execute o notebook na pasta que contém o CSV original."
    )

df = pd.read_csv(
    caminho_base,
    sep=",",
    decimal=",",
    encoding="utf-8",
    quotechar='"',
)

print(f"Arquivo carregado: {caminho_base.name}")


Arquivo carregado: base_churn_5000_VF2_aluno.xlsx - Dados_Sinteticos_VF2.csv


### 2.3. Dimensões da base

Cada linha é um registro e cada coluna é um atributo. A contagem de linhas
não equivale necessariamente à de clientes distintos; a unicidade será examinada
na Parte 3.


In [3]:
n_linhas, n_colunas = df.shape
print(f"Quantidade de linhas: {n_linhas:,}".replace(",", "."))
print(f"Quantidade de colunas: {n_colunas}")


Quantidade de linhas: 5.020
Quantidade de colunas: 10


### 2.4. Primeiras e últimas observações

As cinco primeiras e as cinco últimas linhas ajudam a reconhecer a organização
dos registros e exemplos dos valores armazenados. Esses recortes não são uma
amostra aleatória e não representam, por si só, a distribuição da base.


In [4]:
print("Primeiras cinco observações:")
display(df.head())

print("Últimas cinco observações:")
display(df.tail())


Primeiras cinco observações:


,id_cliente,data_aquisicao_cliente,mensalidade_rs,data_chamado,atrasos_6m,tipo_contrato,forma_pagamento,fibra_optica,debito_automatico,data_ocorrencia_churn
0,C0001,21/04/2024,67.96,"09/09/2026, 15/08/2026",0.0,Bienal,Boleto,0,0,NaN
1,C0002,21/02/2025,65.95,NaN,0.0,Mensal,Cartão,0,0,NaN
2,C0003,21/03/2025,100.88,28/08/2026,1.0,Anual,Boleto,1,1,NaN
3,C0004,21/03/2025,94.80,"06/08/2026, 28/07/2026",2.0,Mensal,Cartão,0,1,NaN
4,C0005,21/02/2022,53.68,05/07/2026,2.0,Anual,Pix,0,0,NaN


Últimas cinco observações:


,id_cliente,data_aquisicao_cliente,mensalidade_rs,data_chamado,atrasos_6m,tipo_contrato,forma_pagamento,fibra_optica,debito_automatico,data_ocorrencia_churn
5015,C3618,21/02/2024,85.79,10/07/2026,1.0,Mensal,Cartão,1,1,NaN
5016,C1643,21/11/2025,57.48,"14/08/2026, 14/07/2026, 01/07/2026",2.0,Anual,Boleto,0,0,NaN
5017,C3711,21/09/2025,97.06,19/09/2026,1.0,Anual,Pix,1,0,NaN
5018,C1235,21/01/2024,111.10,"18/08/2026, 08/07/2026",2.0,Mensal,Pix,1,0,20/11/2026
5019,C4645,21/07/2025,62.15,"11/07/2026, 27/06/2026",0.0,Bienal,Boleto,0,0,NaN


### 2.5. Colunas, tipos e significado preliminar

O tipo de armazenamento não determina sozinho a natureza de um atributo.
Por exemplo, os indicadores de fibra óptica e débito automático são armazenados
como inteiros, mas representam categorias binárias. Datas são carregadas como
texto nesta etapa. A tabela reúne todas as colunas na ordem original.


In [5]:
descricoes_atributos = {
    "id_cliente": ("Identificador", "Código de identificação do cliente."),
    "data_aquisicao_cliente": ("Temporal (texto)", "Data de aquisição do serviço."),
    "mensalidade_rs": ("Numérico contínuo", "Valor da mensalidade em reais."),
    "data_chamado": ("Temporal multivalorado (texto)", "Uma ou mais datas de chamados em um campo."),
    "atrasos_6m": ("Numérico discreto", "Contagem de atrasos em seis meses, conforme o nome."),
    "tipo_contrato": ("Categórico nominal", "Modalidade do contrato."),
    "forma_pagamento": ("Categórico nominal", "Forma de pagamento registrada."),
    "fibra_optica": ("Categórico binário", "Indicador de fibra óptica, com valores 0 e 1 observados."),
    "debito_automatico": ("Categórico binário", "Indicador de débito automático, com valores 0 e 1 observados."),
    "data_ocorrencia_churn": ("Temporal / origem do possível alvo", "Data registrada para a ocorrência de churn."),
}

inventario = pd.DataFrame([
    {
        "Atributo": coluna,
        "Tipo no Pandas": str(df[coluna].dtype),
        "Natureza preliminar": descricoes_atributos[coluna][0],
        "Descrição preliminar": descricoes_atributos[coluna][1],
    }
    for coluna in df.columns
])

display(inventario)


,Atributo,Tipo no Pandas,Natureza preliminar,Descrição preliminar
0,id_cliente,object,Identificador,Código de identificação do cliente.
1,data_aquisicao_cliente,object,Temporal (texto),Data de aquisição do serviço.
2,mensalidade_rs,float64,Numérico contínuo,Valor da mensalidade em reais.
3,data_chamado,object,Temporal multivalorado (texto),Uma ou mais datas de chamados em um campo.
4,atrasos_6m,float64,Numérico discreto,"Contagem de atrasos em seis meses, conforme o ..."
5,tipo_contrato,object,Categórico nominal,Modalidade do contrato.
6,forma_pagamento,object,Categórico nominal,Forma de pagamento registrada.
7,fibra_optica,int64,Categórico binário,"Indicador de fibra óptica, com valores 0 e 1 o..."
8,debito_automatico,int64,Categórico binário,"Indicador de débito automático, com valores 0 ..."
9,data_ocorrencia_churn,object,Temporal / origem do possível alvo,Data registrada para a ocorrência de churn.


### 2.6. Grupos preliminares de atributos

Primeiro, identificamos os campos com armazenamento numérico. Depois, separamos
os grupos pelo significado dos atributos. `atrasos_6m` representa uma contagem,
embora tenha sido lida como `float64`; nenhuma conversão é feita nesta etapa.

Esta classificação serve para organizar a EDA, sem selecionar definitivamente
variáveis de entrada para modelos ou modificar os tipos do DataFrame.


In [6]:
colunas_numericas_armazenadas = df.select_dtypes(include=np.number).columns.tolist()
atributos_numericos = ["mensalidade_rs", "atrasos_6m"]
atributos_categoricos_nominais = ["tipo_contrato", "forma_pagamento"]
atributos_binarios = ["fibra_optica", "debito_automatico"]
atributos_categoricos = atributos_categoricos_nominais + atributos_binarios
atributos_temporais = ["data_aquisicao_cliente", "data_chamado", "data_ocorrencia_churn"]
atributo_identificador = "id_cliente"
coluna_origem_alvo = "data_ocorrencia_churn"

print("Armazenamento numérico:", colunas_numericas_armazenadas)
print("Quantidades numéricas:", atributos_numericos)
print("Categóricos (incluindo binários):", atributos_categoricos)
print("Temporais, ainda como texto:", atributos_temporais)
print("Identificador:", atributo_identificador)
print("Origem do possível alvo:", coluna_origem_alvo)


Armazenamento numérico: ['mensalidade_rs', 'atrasos_6m', 'fibra_optica', 'debito_automatico']
Quantidades numéricas: ['mensalidade_rs', 'atrasos_6m']
Categóricos (incluindo binários): ['tipo_contrato', 'forma_pagamento', 'fibra_optica', 'debito_automatico']
Temporais, ainda como texto: ['data_aquisicao_cliente', 'data_chamado', 'data_ocorrencia_churn']
Identificador: id_cliente
Origem do possível alvo: data_ocorrencia_churn


### 2.7. Estatísticas descritivas iniciais

O resumo das quantidades numéricas apresenta contagem de valores disponíveis,
média, desvio padrão, mínimo, quartis e máximo. A mediana corresponde à linha
`50%`. Os indicadores binários ficam fora desse resumo, pois sua interpretação
principal é categórica.

`describe` desconsidera valores ausentes nos cálculos. As estatísticas abaixo
descrevem os valores lidos, sem qualquer tratamento, e não constituem uma análise
de qualidade, distribuição ou outliers.


In [7]:
resumo_numerico = df[atributos_numericos].describe()
display(resumo_numerico)


,mensalidade_rs,atrasos_6m
count,4900.000,4985.000
mean,91.237,1.204
std,283.899,4.036
min,-85.000,0.000
25%,66.980,0.000
50%,85.915,1.000
75%,98.483,2.000
max,9999.000,99.000


Para os atributos categóricos textuais, o resumo informa a quantidade disponível
(`count`), o número de categorias distintas (`unique`), a categoria mais frequente
(`top`) e sua frequência (`freq`). Os rótulos são considerados exatamente como
estão no arquivo, sem padronização. Tabelas completas de frequências ficam para
a análise univariada.


In [8]:
resumo_categorico = df[atributos_categoricos_nominais].describe().T
display(resumo_categorico)


,count,unique,top,freq
tipo_contrato,4994,6,Mensal,2726
forma_pagamento,5000,6,Boleto,1953


**Leitura inicial dos resumos:** a mensalidade média é aproximadamente **R$ 91,24**
e a mediana é aproximadamente **R$ 85,92**. Para `atrasos_6m`, a média é **1,204**
e a mediana é **1**. Esses números são apenas referências iniciais dos valores
registrados; explicações para suas diferenças exigem as próximas etapas.

Nos campos textuais de contrato e pagamento, o resumo contabiliza **6 rótulos
distintos em cada coluna**. Os rótulos mais frequentes são `Mensal` (**2.726**
registros) e `Boleto` (**1.953** registros), respectivamente. Esses resultados
descrevem as grafias originais e não comprovam seis modalidades de negócio.


### 2.8. Identificação preliminar da variável-alvo

Não existe uma coluna binária chamada `churn` no arquivo. O campo
`data_ocorrencia_churn` é a **origem provável do alvo**, pois registra a data do
evento de cancelamento. Os exemplos abaixo mostram seu conteúdo preenchido,
sem criar rótulos ou converter datas.


In [9]:
exemplos_evento = df.loc[
    df[coluna_origem_alvo].notna(),
    [atributo_identificador, coluna_origem_alvo],
].head(3)

display(exemplos_evento)


,id_cliente,data_ocorrencia_churn
5,C0006,30/10/2026
7,C0008,12/11/2026
9,C0010,20/10/2026


Uma data preenchida registra um possível evento de churn; um campo vazio, isoladamente,
não garante que o cliente permaneceu ativo em qualquer horizonte futuro.
Antes de construir um alvo binário, será necessário confirmar o significado dos
vazios, validar as datas e definir a data de referência e a janela de observação.
Esses parâmetros não serão inferidos da data atual nem importados das referências.

A data do cancelamento não deverá ser utilizada como atributo preditor do próprio
churn, pois revela o evento que se deseja prever. Não foi criada uma variável-alvo
definitiva nesta entrega.


### 2.9. Síntese da compreensão inicial e ponto de revisão

A leitura resultou em **5.020 registros e 10 atributos**. Foram identificados
dois atributos que representam quantidades, quatro categóricos (dois nominais
e dois binários), um identificador e três campos temporais, incluindo a origem
provável do alvo.

Quanto ao armazenamento, há **2 colunas `float64`, 2 `int64` e 6 `object`**.
`data_chamado` contém exemplos de várias datas na mesma célula, o que exigirá
cuidado específico em uma análise temporal futura.

A execução até aqui compreende leitura, inspeção e resumos iniciais. Não houve
escrita na base original, exclusão de registros, imputação, padronização,
codificação, normalização, criação de atributos ou treinamento de modelos.

**Ponto de revisão:** a próxima implementação proposta é a Parte 3, sobre qualidade
e consistência. Somente após a revisão serão desenvolvidas essa parte e as análises
de distribuições, valores extremos, relações com o churn e correlações previstas
no roteiro.
